#### Motivations  

Ce notebook fait suite aux résultats du notebook 
Quelle que soit la taille du segment de signal considéré l'estimation du vecteur de RTF est satisfaisante avec la méthode CS-EVD. 

La réduction de la taille de segment utile engendre des erreurs d'estimation sur les matrices de covariance du signal et du bruit. 


Pour la position P1 :

La matrice de covariance du bruit est plus sensible à la taille du segment de signal utilisé. Néanmoins, en raison du RSB élevé (les coefficients de $R_v$ sont deux ordres de grandeurs plus faibles que ceux de $R_x$), les erreurs relatives d'estimation de $R_v$ importantes (entre 18.9 et 93.2 %) entrainent de relativement faibles erreurs d'estimation de $R_z$ même pour des signaux très courts (1 à 2 s).

Pour T >= 3s, l'erreur relative absolue $\Delta R_z$ moyenne sur la bande de fréquence représentée ci-dessus est inférieure à 3%.  
Pour la plus petite fenêtre considérée (T = 1s), l'erreur relative absolue $\Delta R_z$ moyenne sur la bande de fréquence représentée ci-dessus est inférieure à 10%. 

Des conclusions similaires peuvent être dressées pour les autres positions de la source (P2 à P6) :

* Erreurs d'estimation de $R_v$ importantes pour T petit. 
* Faible impact sur l'estimation de $R_z$ en raison du très fort RSB. 

**Remarque sur les valeurs propres de Rz** : 

Rz est une matrice de covariance, elle est donc hermitienne semi-définie positive. Ainsi, toutes ces valeurs propres sont réelles et positives ou nulles. 

Ici certaines des valeurs propres peuvent être négatives en raison des erreurs d'estimation. En effet, les matrices de covariance Rx et Rv ne sont pas nécessairement Hermitienne semi-définie positive en raison de ces erreurs d'estimation. 

De plus, 

$
R_z = \Phi_1 \Pi \Pi^H
$
et donc Rz est de rang 1. 

Puisque Rz est Hermitienne elle est diagonalisable et donc semblable à la matrice D (matrice dans la base de diagonalisation)

$
D = U^H R_z U
$

Le rang de Rz est donc égal au rang de D, c'est-à-dire au nombre de valeurs propres non nulles de Rz. 

En pratique, en raison des erreurs d'estimation les valeurs propres de Rz ne sont pas nulles, néanmoins, on peut vérifier que la valeur propre principale de la matrice Rz est largement supérieure aux autres valeurs propres. Ici le rapport de la valeur propre principale aux autres valeurs propres est au moins d'un ordre de grandeur. 

**CONCLUSION** : 

Le test ci-dessus ne permet pas d'expliquer de manière satisfaisante les faibles performances de localisation obtenues avec la méthode CS-EVD dans le cas de la localisation de la source en mouvement. 

Il faut étudier les performances des estimateurs sur le signal de la source en mouvement au passage de l'un des points statiques de référence. 

Lors de l'essai en dynamique le chariot roule le long du bassin et est susceptible de générer un niveau de bruit plus important. On dispose néanmoins d'un enregistrement de roulage à vide (sans émission de la source que l'on peut exploiter pour obtenir des estimations plus fiables de la matrice de covariance du bruit). On peut par exemple exploiter des fenêtres de 3s, 1s avant et après la tranche de signal d'intérêt.  

In [1]:
import os 
import sys 
import numpy as np 
import xarray as xr
import scipy.signal as sp
import matplotlib.pyplot as plt  

sys.path.append(r"C:\Users\baptiste.menetrier\Desktop\devPy\phd")

import params
from time import time
from scipy.linalg import eigh
from scipy.optimize import least_squares

from utils import load_fiberscope_data
from fiberscope_manager import FiberscopeManager
from fiberscope_recording import FiberscopeSweep1, FiberscopeSweep2, FiberscopeDynamicRecording
from propa.rtf.rtf_utils import D_hermitian_angle_fast, normalize_metric_contrast

In [2]:
# Useful properties
h_index_ref = 5     # On choisit le récepteur avec le plus haut SNR 

# Flags 
run_static_demo = False

In [3]:
ds_rtf = xr.open_dataset(
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\fiberscope_20\data\dynamic\10-10-2024T16-53-43-200271_PR_N1_346.nc"
)
fs = ds_rtf.fs

In [4]:
ns_opti = 2**16
alpha_ov_opti = 0.75

T_stat = 14.6
T_dyn = 3       # Par exemple 

def L_available_seg(T, fs, n_stft, alpha_ov):
    L = (T * fs - n_stft + alpha_ov * n_stft) / (alpha_ov * n_stft) 
    return np.floor(L) 

L_dyn = L_available_seg(T=T_dyn, fs=fs, n_stft=ns_opti, alpha_ov=alpha_ov_opti) 
L_stat = L_available_seg(T=T_stat, fs=fs, n_stft=ns_opti, alpha_ov=alpha_ov_opti)

print(f"L_dyn = {L_dyn}")
print(f"L_stat = {L_stat}")

def n_stft_dyn_cst_L(T, fs, L_ref, alpha_ov_ref):
    n_cst_L = (T * fs) / (1 + alpha_ov_ref * (L_ref - 1))
    n_stft_cst_L = 2 ** (np.floor(np.log2(n_cst_L)) + 1)
    return n_stft_cst_L

n_stft_dyn = n_stft_dyn_cst_L(T=T_dyn, fs=ds_rtf.fs, L_ref=L_stat, alpha_ov_ref=alpha_ov_opti)
print(f"n_stft for dynamic recording : 2**{np.log2(n_stft_dyn)}")

L_dyn = 2.0
L_stat = 11.0
n_stft for dynamic recording : 2**14.0


In [4]:
# Define the order based on the source position : from closest to farthest
dict_th_pos = params.dict_th_pos
all_dists = np.array([dict_th_pos[pos] for pos in dict_th_pos.keys()])
idx_pos_sort = np.argsort(all_dists)
src_labels = list(dict_th_pos.keys())
src_label_sorted = [src_labels[i] for i in idx_pos_sort]
all_dists_sorted = all_dists[idx_pos_sort]

In [6]:
# Fiberscope dynamic recording
fs_dynamic = FiberscopeDynamicRecording()

# Create an instance of FiberscopeManager
root_processed_data = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\real_data_analysis\fiberscope_20\data\dynamic\test_cs_evd_perf"
if not os.path.exists(root_processed_data):
    os.makedirs(root_processed_data)

fsm = FiberscopeManager(
    root_processed_data=root_processed_data,
    h_index_ref=h_index_ref,
    plot_feature=False,
)

fs_sweep1 = FiberscopeSweep1()
fs_sweep1.records_folder = os.path.join(root_processed_data, "static")
if not os.path.exists(fs_sweep1.records_folder):
    os.makedirs(fs_sweep1.records_folder)

records_to_process = fs_sweep1.records_N1
results = {recording_name: {"rtf_ref_static": None, 
                            "rtf_cs_evd_static": None, 
                            "rtf_ref_dynamic": [],
                            "rtf_cs_dynamic": [], 
                            "rtf_cs_evd_dynamic": [],
                            } for recording_name in records_to_process}

In [7]:
# # Load and preprocess static record
# nperseg = ns_opti
# noverlap = int(nperseg * alpha_ov_opti)
# fsm.nperseg = nperseg
# fsm.noverlap = noverlap
# rtf_estimator = "cs-evd"

# fsm.process_static_analysis(
#     static_signal=fs_sweep1,
#     static_records_names=records_to_process,
#     set_stft_props=True,
#     rtf_estimator=rtf_estimator,
# )

In [5]:
# Iterate over static files
for recording_name in records_to_process:
    # Load precomputed RTF
    fpath = os.path.join(fs_sweep1.records_folder, recording_name) + "_rtf.nc"
    xr_data = xr.open_dataset(fpath)

    # Store reference RTF vector for each source position (ie each static record)
    results[recording_name]["rtf_ref_static"] = xr_data.rtf_amp * np.exp(
        1j * xr_data.rtf_phase
    )
    results[recording_name]["rtf_cs_evd_static"] = xr_data.rtf_amp_hat * np.exp(
        1j * xr_data.rtf_phase_hat
    )

NameError: name 'records_to_process' is not defined

In [9]:
# time_steps = [1, 2, 3, 4, 5]  # s
time_steps = [3]  # s

# Get the time at which the moving source crosses a known static position using estimated range vs time from TOA estimation
alpha = 0.0999
beta = -0.0513
# rt_toa = alpha * ds_rtf.time.values + beta
# ti = (ri - beta) / alpha
theoretical_ti = {k: (dict_th_pos[k] - beta) / alpha for k in dict_th_pos.keys()}

# Iterate over time steps
for time_step in time_steps:
    fs_dynamic = FiberscopeDynamicRecording()

    # Mise à jour des paramètres stft à utiliser en fonction du pas de temps
    n_sweep_step = int(time_step / fs_dynamic.signal.interp_pulse_period)

    n_stft_dyn = n_stft_dyn_cst_L(
        T=time_step, fs=fs, L_ref=L_stat, alpha_ov_ref=alpha_ov_opti
    )
    n_stft_dyn = n_stft_dyn
    alpha_ov_dyn = alpha_ov_opti
    print(f"n_stft for dynamic recording : 2**{np.log2(n_stft_dyn)}")

    # Set stft params for dynamic records
    nperseg = n_stft_dyn
    noverlap = int(nperseg * alpha_ov_dyn)
    fsm.nperseg = nperseg
    fsm.noverlap = noverlap

    # fsm.set_stft_params(ts=ds_rtf.ts)
    print(f"(nperseg = 2**{np.log2(fsm.nperseg)}, nov = {fsm.noverlap})")

    # preSplit the dynamic recording
    fsm.presplit_dynamic_record(
        fs_dynamic_recording=fs_dynamic,
        n_sweep=n_sweep_step,
        t_max=np.max(ds_rtf.time.values),
    )

    #  Split dynamic records and save as nc 
    need_to_compute = not(len(os.listdir(fs_dynamic.splitted_records_folder)) // len(list(dict_th_pos.keys())) in [1, 2])
    if need_to_compute:
        fsm.split_dynamic_record(fs_dynamic_recording=fs_dynamic, force_reload=False)

        # Extract ranges corresponding to each splitted record
        splitted_record_ranges = [float(fname.split("_")[-2].split("r")[1].split("m")[0]) for fname in fs_dynamic.splitted_records_names]
        # Find the record that is the closest to each theoretical position
        records_close_to_static_pos =[fs_dynamic.splitted_records_names[idx] for idx in [np.argmin(np.abs(np.array(splitted_record_ranges) - dict_th_pos[k])) for k in dict_th_pos.keys()]]
        # Keep only the records of interest
        fs_dynamic.splitted_records_names = records_close_to_static_pos
        # Delete other records from folder
        for fname in os.listdir(fs_dynamic.splitted_records_folder):
            fname_ = fname.split(".nc")[0]  # Remove extension
            if fname_ not in records_close_to_static_pos:
                os.remove(os.path.join(fs_dynamic.splitted_records_folder, fname))
    else:
        fs_dynamic.splitted_records_names = [fname.split(".nc")[0] for fname in os.listdir(fs_dynamic.splitted_records_folder)]

    # Derive features for CS-EVD
    rtf_estimator = "cs-evd"
    fsm.process_dyn_analysis(
        fs_dynamic_recording=fs_dynamic,
        use_global_noise_csdm=False,
        set_stft_props=False,
        rtf_estimator=rtf_estimator,
    )

    # Store results
    for recording_name in fs_dynamic.splitted_records_names:
        # Load precomputed RTF
        fpath = os.path.join(fs_dynamic.splitted_records_folder, recording_name) + "_rtf.nc"
        xr_data = xr.open_dataset(fpath)

        # Store estimated RTF vector to the corresponding static position (ie each dynamic record)
        corresponding_static_pos = min(dict_th_pos.keys(), key=lambda k: abs(dict_th_pos[k] - float(recording_name.split("_")[-2].split("r")[1].split("m")[0])))
        recording_name_static_pos = [rec for rec in records_to_process if corresponding_static_pos in rec][0]
        results[recording_name_static_pos]["rtf_cs_evd_dynamic"].append(
            xr_data.rtf_amp_hat * np.exp(1j * xr_data.rtf_phase_hat)
        )
        results[recording_name_static_pos]["rtf_ref_dynamic"].append(
            xr_data.rtf_amp * np.exp(1j * xr_data.rtf_phase)
        )
        xr_data.close()
        del xr_data

    # Same for CS (not EVD)
    rtf_estimator = "cs"
    fsm.process_dyn_analysis(
        fs_dynamic_recording=fs_dynamic,
        use_global_noise_csdm=False,
        set_stft_props=False,
        rtf_estimator=rtf_estimator,
    )
    
    # Store results
    for recording_name in fs_dynamic.splitted_records_names:
        # Load precomputed RTF
        fpath = os.path.join(fs_dynamic.splitted_records_folder, recording_name) + "_rtf.nc"
        xr_data = xr.open_dataset(fpath)

        # Store estimated RTF vector to the corresponding static position (ie each dynamic record)
        corresponding_static_pos = min(
            dict_th_pos.keys(),
            key=lambda k: abs(
                dict_th_pos[k]
                - float(recording_name.split("_")[-2].split("r")[1].split("m")[0])
            ),
        )
        recording_name_static_pos = [rec for rec in records_to_process if corresponding_static_pos in rec][0]
        results[recording_name_static_pos]["rtf_cs_dynamic"].append(
            xr_data.rtf_amp_hat * np.exp(1j * xr_data.rtf_phase_hat)
        )

        xr_data.close()
        del xr_data
    print("-----------------------------------")

n_stft for dynamic recording : 2**14.0
(nperseg = 2**14.0, nov = 12288)


IndexError: list index out of range

In [ ]:
results

In [ ]:
dist_func = D_hermitian_angle_fast
ax_f = 1
ax_rcv = 0
dist_kwargs = {
    "ax_f": ax_f,
    "unit": "deg",
    "ax_rcv": ax_rcv,
    "apply_mean": False,
    "apply_median": True,
}

In [ ]:
# Compute theta at each position between cs and cs-evd (dynamic)
for recording_name in records_to_process:
    print(f"Processing results for static record {recording_name}")
    res = results[recording_name]
    rtf_ref_static = res["rtf_ref_static"]
    rtf_cs_evd_static = res["rtf_cs_evd_static"]

    # Restrict to common frequency band
    fmin = fs_dynamic.signal.fmin
    fmax = fs_dynamic.signal.fmax
    fmin = 10500
    fmax = 10600
    rtf_ref_static = rtf_ref_static.sel(f_ir=slice(fmin, fmax))
    rtf_cs_evd_static = rtf_cs_evd_static.sel(f_rtf=slice(fmin, fmax))

    rtf_ref_dynamic = res["rtf_ref_dynamic"]
    rtf_cs_dynamic = res["rtf_cs_dynamic"]
    rtf_cs_evd_dynamic = res["rtf_cs_evd_dynamic"]

    for k in range(len(rtf_cs_dynamic)):
        print(f"Time step = {time_steps[k]} s")
        rtf_cs_dyn_k = rtf_cs_dynamic[k].sel(f_rtf=slice(fmin, fmax))
        rtf_cs_evd_dyn_k = rtf_cs_evd_dynamic[k].sel(f_rtf=slice(fmin, fmax))
        rtf_ref_dyn_k = rtf_ref_dynamic[k].sel(f_ir=slice(fmin, fmax))

        # Interp at the same freq bins
        rtf_ref_static_int = rtf_ref_static.sel(f_ir=rtf_cs_dyn_k.f_rtf, method="nearest")
        rtf_ref_dyn_k_int = rtf_ref_dyn_k.sel(f_ir=rtf_cs_dyn_k.f_rtf, method="nearest")
        theta_cs_ref = dist_func(
            rtf_ref_static_int.values,
            rtf_cs_dyn_k.values,
            **dist_kwargs,
        )
        theta_cs_ref_dyn = dist_func(
            rtf_ref_dyn_k_int.values,
            rtf_cs_dyn_k.values,
            **dist_kwargs,
        )

        theta_cs_evd_ref = dist_func(
            rtf_ref_static_int.values,
            rtf_cs_evd_dyn_k.values,
            **dist_kwargs,
        )
        theta_cs_evd_ref_dyn = dist_func(
            rtf_ref_dyn_k_int.values,
            rtf_cs_evd_dyn_k.values,
            **dist_kwargs,
        )

        theta_cs_evd_cs = dist_func(
            rtf_cs_dyn_k.values,
            rtf_cs_evd_dyn_k.values,
            **dist_kwargs,
        )
        print(f"theta(cs, ref static) = {theta_cs_ref:.2f} deg")
        print(f"theta(cs-evd, ref static) = {theta_cs_evd_ref:.2f} deg")
        print(f"theta(cs, ref dynamic) = {theta_cs_ref_dyn:.2f} deg")
        print(f"theta(cs-evd, ref dynamic) = {theta_cs_evd_ref_dyn:.2f} deg")
        print(f"theta(cs, cs-evd) = {theta_cs_evd_cs:.2f} deg")
        print("-----")

In [ ]:
# Plot for each static position the RTF estimates from static and dynamic records
for recording_name in records_to_process:
    res = results[recording_name]
    rtf_ref_static = res["rtf_ref_static"]
    rtf_cs_evd_static = res["rtf_cs_evd_static"]
    rtf_ref_dynamic = res["rtf_ref_dynamic"]
    rtf_cs_dynamic = res["rtf_cs_dynamic"]
    rtf_cs_evd_dynamic = res["rtf_cs_evd_dynamic"]

    # nrcv = rtf_ref_static.sizes["h_index"]
    nrcv = 2  # For better visualization
    f_amp, axs_amp = plt.subplots(nrows=nrcv, ncols=1, sharex=True)
    f_amp.suptitle(f"RTF estimates for static record {recording_name}")
    # f_phase, axs_phase = plt.subplots(nrows=nrcv, ncols=1, sharex=True)

    # Restrict to common frequency band
    fmin = fs_dynamic.signal.fmin
    fmax  = fs_dynamic.signal.fmax
    fmin = 10500
    fmax = 11500
    rtf_ref_static = rtf_ref_static.sel(f_ir=slice(fmin, fmax))
    rtf_cs_evd_static = rtf_cs_evd_static.sel(f_rtf=slice(fmin, fmax))
    rtf_ref_static_amp = np.abs(rtf_ref_static)
    rtf_ref_static_phase = np.angle(rtf_ref_static)

    rtf_cs_evd_static_amp = np.abs(rtf_cs_evd_static)
    rtf_cs_evd_static_phase = np.angle(rtf_cs_evd_static)

    for i, idx_rcv in enumerate(rtf_ref_static.h_index.values[0:nrcv]):

        ## Amplitudes ##
        # Plot ref rtf from static record (deconvolution)
        rtf_ref_static_amp.sel(h_index=idx_rcv).plot(
            ax=axs_amp[i], color="k", label=f"Ref-static-deconvolution {idx_rcv}"
        )
        # Plot CS-EVD rtf from static record
        rtf_cs_evd_static_amp.sel(h_index=idx_rcv).plot(
            ax=axs_amp[i],
            color="g",
            marker="o",
            markersize=1,
            linewidth=1,
            linestyle="--",
            label=f"Ref-static-CSEVD - {idx_rcv}",
        )
        # # Plot ref rtf from dynamic records (deconvolution)
        # for j, rtf_ref_dyn in enumerate(rtf_ref_dynamic):
        #     rtf_ref_dyn_amp = np.abs(rtf_ref_dyn).sel(f_ir=slice(fmin, fmax))
        #     rtf_ref_dyn_amp.sel(h_index=idx_rcv).plot(
        #         ax=axs_amp[i],
        #         marker=".",
        #         markersize=1,
        #         linewidth=1,
        #         linestyle=":",
        #         label=f"Ref-dyn-deconvolution -{j+1} - {idx_rcv}",
        #     )

        # Plot CS-EVD rtf from dynamic records
        for j, rtf_cs_evd_dyn in enumerate(rtf_cs_evd_dynamic[0:1]):
            rtf_cs_evd_dyn_amp = np.abs(rtf_cs_evd_dyn).sel(f_rtf=slice(fmin, fmax))
            rtf_cs_evd_dyn_amp.sel(h_index=idx_rcv).plot(
                ax=axs_amp[i],
                marker="x",
                markersize=1,
                linewidth=1,
                linestyle="-",
                label=f"Ref-dyn-CSEVD -{j+1} - {idx_rcv}",
            )
        # Plot CS rtf from dynamic records
        for j, rtf_cs_dyn in enumerate(rtf_cs_dynamic[0:1]):
            rtf_cs_dyn_amp = np.abs(rtf_cs_dyn).sel(f_rtf=slice(fmin, fmax))
            rtf_cs_dyn_amp.sel(h_index=idx_rcv).plot(
                ax=axs_amp[i],
                marker="o",
                markersize=1,
                linewidth=1,
                linestyle="--",
                label=f"Ref-dyn-CS -{j+1} - {idx_rcv}",
            )

        axs_amp[i].set_xlabel("")
        axs_amp[i].set_ylabel(r"$|\Pi|$")
        axs_amp[i].set_yscale("log")
        axs_amp[i].set_title("")
        axs_amp[i].legend(fontsize=8)

        # ## Phases ##
        # # Plot ref rtf from static record (deconvolution)
        # rtf_ref_static_phase.sel(h_index=idx_rcv).plot(
        #     ax=axs_phase[i], color="k", label=f"Ref-static-deconvolution {idx_rcv}"
        # )
        # # Plot CS-EVD rtf from static record
        # rtf_cs_evd_static_phase.sel(h_index=idx_rcv).plot(
        #     ax=axs_phase[i],
        #     color="g",
        #     marker="o",
        #     markersize=1,
        #     linewidth=1,
        #     linestyle="--",
        #     label=f"Ref-static-CSEVD - {idx_rcv}",
        # )
        # axs_phase[i].set_xlabel("")
        # axs_phase[i].set_ylabel(r"$\Phi$")
        # axs_phase[i].set_title("")
        # axs_phase[i].legend(fontsize=8)